## 1. Ingestion

In [ ]:
import os, json, time, requests
import pandas as pd
from dotenv import load_dotenv

# https://pypi.org/project/python-dotenv/
# jsonrpc.org/specification
load_dotenv(dotenv_path="../.env")
URL = f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv('ALCHEMY_KEY')}"

def rpc(method, params, retries=3):
    for attempt in range(retries):
        try:
            r = requests.post(URL, 
                              json={"jsonrpc": "2.0",
                                    "id": 1, "method": method,
                                    "params": params},
                                    timeout=30)
            result = r.json()
            if "error" in result:
                raise Exception(result["error"])
            return result["result"]
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  retry {attempt+1} after error: {e}")
            time.sleep(2 ** attempt)

latest = int(rpc("eth_blockNumber", []), 16)
print("Latest block:", latest)

Latest block: 25110982


In [22]:
# Define fixed range — hardcode for reproducibility
BUFFER, WINDOW = 10, 1000
END_BLOCK = latest - BUFFER
START_BLOCK = END_BLOCK - (WINDOW-1)
print(f"Pulling blocks {START_BLOCK} to {END_BLOCK} ({WINDOW} blocks)")

Pulling blocks 25109973 to 25110972 (1000 blocks)


In [23]:
# Pull all blocks with full transactions
to, tx_rows = time.time(), []

for n in range(START_BLOCK, END_BLOCK + 1):
    cache_path = f"../data/raw/block_{n}.json"
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            b = json.load(f)
    else:
        b = rpc("eth_getBlockByNumber", [hex(n), True])
        with open(cache_path, "w") as f:
            json.dump(b, f)
        time.sleep(0.05)

    ts = int(b["timestamp"], 16)
    for tx in b["transactions"]:
        tx_rows.append({
            "block": n,
            "ts": ts,
            "tx_hash": tx["hash"],
            "from": tx["from"],
            "to": tx["to"],
            "value_wei": int(tx["value"], 16),
            "gas": int(tx["gas"], 16),
            "input_len": len(tx["input"]),
            "is_contract_call": tx["input"] != "0x",
        })

    if (n - START_BLOCK) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  block {n} ({n - START_BLOCK + 1}/{END_BLOCK - START_BLOCK + 1}) — {elapsed:.0f}s elapsed")

print(f"\nDone. {len(tx_rows)} transactions in {time.time()-t0:.0f}s")

  block 25109973 (1/1000) — 254s elapsed
  block 25110023 (51/1000) — 254s elapsed
  block 25110073 (101/1000) — 254s elapsed
  block 25110123 (151/1000) — 255s elapsed
  block 25110173 (201/1000) — 255s elapsed
  block 25110223 (251/1000) — 255s elapsed
  block 25110273 (301/1000) — 255s elapsed
  block 25110323 (351/1000) — 255s elapsed
  block 25110373 (401/1000) — 255s elapsed
  block 25110423 (451/1000) — 255s elapsed
  block 25110473 (501/1000) — 255s elapsed
  block 25110523 (551/1000) — 255s elapsed
  block 25110573 (601/1000) — 255s elapsed
  block 25110623 (651/1000) — 255s elapsed
  block 25110673 (701/1000) — 255s elapsed
  block 25110723 (751/1000) — 256s elapsed
  block 25110773 (801/1000) — 256s elapsed
  block 25110823 (851/1000) — 256s elapsed
  block 25110873 (901/1000) — 256s elapsed
  block 25110923 (951/1000) — 256s elapsed

Done. 388647 transactions in 256s


In [14]:
# Build transactions DataFrame, save
tx_df = pd.DataFrame(tx_rows)
tx_df["value_eth"] = tx_df["value_wei"] / 1e18
tx_df["dt"] = pd.to_datetime(tx_df["ts"], unit="s")

# tx_df.to_parquet("../data/processed/transactions.parquet")
tx_df.to_csv("../data/processed/transactions.csv", index=False)
print(f"Saved {len(tx_df):,} txs")
print(f"Unique senders: {tx_df['from'].nunique():,}")
print(f"Unique recipients: {tx_df['to'].nunique():,}")
print(f"Contract call ratio: {tx_df['is_contract_call'].mean():.2%}")
tx_df.head()

Saved 388,647 txs
Unique senders: 126,024
Unique recipients: 59,715
Contract call ratio: 39.90%


,block,ts,tx_hash,from,to,value_wei,gas,input_len,is_contract_call,value_eth,dt
0,25109973,1778961467,0xe776f66f0927a53d7c68a6880051bd92cd1c0eba33ee...,0xc917c3fa468f2c4b9c84c72caa46420eb9825249,0x0000000aa232009084bd71a5797d089aa4edfad4,0,378092,1290,True,0.0,2026-05-16 19:57:47
1,25109973,1778961467,0xcdc82cda158ab0115b256bfb29ae23132a30b67e8bd8...,0xae2fc483527b8ef99eb5d9b44875f005ba1fae13,0x1f2f10d1c40777ae1da742455c65828ff36df387,33867796283,148650,98,True,0.0,2026-05-16 19:57:47
2,25109973,1778961467,0xb04209e2045951d8eb9f69e92aea8f2d4025fdb3de42...,0x73fac1a45efa0f6ad8481d0252a500754c732ba0,0xb300000b72deaeb607a12d5f54773d1c19c7028d,0,1593074,9802,True,0.0,2026-05-16 19:57:47
3,25109973,1778961467,0xb7b98fd77e26eaa68e1abb16ed94e780e74e3e997dbe...,0xae2fc483527b8ef99eb5d9b44875f005ba1fae13,0x1f2f10d1c40777ae1da742455c65828ff36df387,33862539579,140294,98,True,0.0,2026-05-16 19:57:47
4,25109973,1778961467,0x98d9f968f4fe82f51ba78d351015c03c94f368c1f769...,0xe5402d088d5f051219c84aa7c221fdaaa889c803,0x681e908b8ab57c49c74d770f369754ccc3e1ae09,0,35000,74,True,0.0,2026-05-16 19:57:47


In [16]:
# Pull ERC-20 Transfer logs, chunked (Alchemy caps log queries)
TRANSFER_TOPIC = "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"
CHUNK = 10  # 100 blocks per getLogs call — safely under any range limit

all_logs = []
t0 = time.time()

for chunk_start in range(START_BLOCK, END_BLOCK + 1, CHUNK):
    chunk_end = min(chunk_start + CHUNK - 1, END_BLOCK)
    logs = rpc("eth_getLogs", [{
        "fromBlock": hex(chunk_start),
        "toBlock":   hex(chunk_end),
        "topics":    [TRANSFER_TOPIC],
    }])
    all_logs.extend(logs)
    print(f"  blocks {chunk_start}-{chunk_end}: {len(logs)} logs (total: {len(all_logs):,})")
    time.sleep(0.05)

print(f"\nDone. {len(all_logs):,} Transfer logs in {time.time()-t0:.0f}s")

  blocks 25109973-25109982: 4797 logs (total: 4,797)
  blocks 25109983-25109992: 4801 logs (total: 9,598)
  blocks 25109993-25110002: 5155 logs (total: 14,753)
  blocks 25110003-25110012: 5316 logs (total: 20,069)
  blocks 25110013-25110022: 4849 logs (total: 24,918)
  blocks 25110023-25110032: 4650 logs (total: 29,568)
  blocks 25110033-25110042: 4877 logs (total: 34,445)
  blocks 25110043-25110052: 4834 logs (total: 39,279)
  blocks 25110053-25110062: 4623 logs (total: 43,902)
  blocks 25110063-25110072: 4245 logs (total: 48,147)
  blocks 25110073-25110082: 5152 logs (total: 53,299)
  blocks 25110083-25110092: 5153 logs (total: 58,452)
  blocks 25110093-25110102: 4265 logs (total: 62,717)
  blocks 25110103-25110112: 4802 logs (total: 67,519)
  blocks 25110113-25110122: 4743 logs (total: 72,262)
  blocks 25110123-25110132: 4791 logs (total: 77,053)
  blocks 25110133-25110142: 4426 logs (total: 81,479)
  blocks 25110143-25110152: 5561 logs (total: 87,040)
  blocks 25110153-25110162: 48

In [15]:
# Decode logs into a clean DataFrame
def topic_to_addr(topic):
    # topics are 32 bytes; addresses are last 20 bytes (40 hex chars)
    return "0x" + topic[-40:].lower()

log_rows = []
for log in all_logs:
    # ERC-721 Transfers also use the same topic but have 4 topics (the 4th is tokenId).
    # ERC-20 has exactly 3 topics. Filter to ERC-20 only.
    if len(log["topics"]) != 3:
        continue
    log_rows.append({
        "block": int(log["blockNumber"], 16),
        "tx_hash": log["transactionHash"],
        "log_index": int(log["logIndex"], 16),
        "token": log["address"].lower(),
        "from": topic_to_addr(log["topics"][1]),
        "to":   topic_to_addr(log["topics"][2]),
        "value_raw": int(log["data"], 16) if log["data"] != "0x" else 0,
    })

logs_df = pd.DataFrame(log_rows)
# attach timestamp via block → ts mapping from tx_df
block_ts = tx_df.drop_duplicates("block").set_index("block")["ts"]
logs_df["ts"] = logs_df["block"].map(block_ts)
logs_df["dt"] = pd.to_datetime(logs_df["ts"], unit="s")

logs_df.to_csv("../data/processed/token_transfers.csv", index=False)
print(f"Saved {len(logs_df):,} ERC-20 transfers")
print(f"Unique tokens: {logs_df['token'].nunique():,}")
print(f"Unique senders: {logs_df['from'].nunique():,}")
print(f"Unique recipients: {logs_df['to'].nunique():,}")
logs_df.head()

NameError: name 'all_logs' is not defined